In [1]:
pip install -r requirements_dev.txt

  Using cached typing_extensions-4.6.3-py3-none-any.whl.metadata (2.8 kB)
Using cached typing_extensions-4.6.3-py3-none-any.whl (31 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx 1.20.0 requires typing_extensions>=4.7.1, but you have typing-extensions 4.6.3 which is incompatible.
torch 2.5.1+cu121 requires typing-extensions>=4.8.0, but you have typing-extensions 4.6.3 which is incompatible.


In [2]:
!pip install smartapi-python pyotp pandas openpyxl

In [3]:
pip install pyotp

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install logzero

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install websocket-client

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip uninstall pycrypto

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install pycryptodome

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install SmartApi

  Using cached SmartAPI-1.1.0-py3-none-any.whl
  Using cached pycrypto-2.6.1.tar.gz (446 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build PyCrypto
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for PyCrypto (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      C:\Users\Kuro Gaming\AppData\Local\Temp\pip-build-env-u5d_7jvk\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: Public Domain
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudi

In [9]:
!pip install watchdog

In [10]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [12]:
# 🚀 DAY 42: ANGEL ONE LIVE WEBSOCKET
print("🎯 Tick → GPU Pipeline | Day41 Risk → Market Data")
print("Production datafeed")

from SmartApi import SmartConnect
from SmartApi.smartWebSocketV2 import SmartWebSocketV2
from pyotp import TOTP
import threading
import queue
import time
from datetime import datetime

# =========================
# ANGEL ONE CREDENTIALS
# =========================
API_KEY = "hYMUaSKT"
CLIENT_CODE = "AAAQ167458"
MPIN_OR_PASSWORD = "9161"
TOTP_SECRET = "IIBHSWOSJYPGFBWM75R6NHSWTI"

# =========================
# DAY41 POSITIONS / WATCHLIST
# =========================
watchlist = ['HCLTECH', 'KOTAKBANK', 'RELIANCE', 'HDFCBANK']

# NSE cash tokens
TOKEN_MAP = {
    "HCLTECH": "7229",
    "KOTAKBANK": "1922",   # verify once from your token master before production
    "RELIANCE": "2885",
    "HDFCBANK": "1333"
}

REVERSE_TOKEN_MAP = {v: k for k, v in TOKEN_MAP.items()}

data_queue = queue.Queue()
tick_count = 0
processed_count = 0
MAX_TICKS = 10

stop_event = threading.Event()

# =========================
# LOGIN
# =========================
obj = SmartConnect(api_key=API_KEY)
session = obj.generateSession(
    CLIENT_CODE,
    MPIN_OR_PASSWORD,
    TOTP(TOTP_SECRET).now()
)

if not session.get("status"):
    raise Exception(f"Login failed: {session}")

AUTH_TOKEN = session["data"]["jwtToken"]
FEED_TOKEN = obj.getfeedToken()

# =========================
# WEBSOCKET
# =========================
correlation_id = "day42live"
mode = 1  # 1 = LTP
token_list = [{
    "exchangeType": 1,  # 1 = NSE CM
    "tokens": list(TOKEN_MAP.values())
}]

sws = SmartWebSocketV2(
    AUTH_TOKEN,
    API_KEY,
    CLIENT_CODE,
    FEED_TOKEN,
    max_retry_attempt=3
)

def on_data(wsapp, message):
    global tick_count

    try:
        token = message.get("token")
        symbol = REVERSE_TOKEN_MAP.get(token, token)

        if symbol in watchlist:
            ltp = message.get("last_traded_price", 0) / 100.0
            volume = message.get("volume_trade_for_the_day", 0)
            exch_ts = message.get("exchange_timestamp")

            tick_count += 1
            data_queue.put({
                "symbol": symbol,
                "ltp": ltp,
                "volume": volume,
                "exchange_timestamp": exch_ts,
                "time": datetime.now().strftime("%H:%M:%S")
            })

            print(f"📡 LIVE TICK #{tick_count}: {symbol} ₹{ltp:.2f}")

            if tick_count >= MAX_TICKS:
                stop_event.set()
                try:
                    sws.close_connection()
                except Exception:
                    pass

    except Exception as e:
        print("on_data error:", e)

def on_open(wsapp):
    print("✅ WebSocket opened")
    sws.subscribe(correlation_id, mode, token_list)
    print(f"📡 Subscribed to: {', '.join(watchlist)}")

def on_error(wsapp, error):
    print("❌ WebSocket error:", error)
    stop_event.set()

def on_close(wsapp):
    print("🔌 WebSocket closed")

sws.on_open = on_open
sws.on_data = on_data
sws.on_error = on_error
sws.on_close = on_close

def start_feed():
    sws.connect()

# Start live feed
feed_thread = threading.Thread(target=start_feed, daemon=True)
feed_thread.start()

# =========================
# PROCESS QUEUE (GPU PIPELINE)
# =========================
start_time = time.time()
TIMEOUT_SECONDS = 120

while processed_count < MAX_TICKS and not stop_event.is_set():
    try:
        tick = data_queue.get(timeout=1.0)
        processed_count += 1

        print(
            f"→ GPU QUEUE: {tick['symbol']} @ ₹{tick['ltp']:.2f} | "
            f"Vol:{tick['volume']}"
        )

        data_queue.task_done()

    except queue.Empty:
        if time.time() - start_time > TIMEOUT_SECONDS:
            print("⏰ Timeout waiting for ticks")
            stop_event.set()
            try:
                sws.close_connection()
            except Exception:
                pass
            break

print("\n🎯 DAY 42: Live datafeed production!")
print(f"• Received ticks: {tick_count}")
print(f"• Processed ticks: {processed_count}")
print("• GitHub Day42 → 42/90")

🎯 Tick → GPU Pipeline | Day41 Risk → Market Data
Production datafeed


[I 260331 22:18:23 smartConnect:121] in pool


✅ WebSocket opened
📡 Subscribed to: HCLTECH, KOTAKBANK, RELIANCE, HDFCBANK
📡 LIVE TICK #1: RELIANCE ₹1343.90→ GPU QUEUE: RELIANCE @ ₹1343.90 | Vol:0

📡 LIVE TICK #2: HDFCBANK ₹731.55→ GPU QUEUE: HDFCBANK @ ₹731.55 | Vol:0

📡 LIVE TICK #3: HCLTECH ₹1341.60→ GPU QUEUE: HCLTECH @ ₹1341.60 | Vol:0

📡 LIVE TICK #4: KOTAKBANK ₹353.40→ GPU QUEUE: KOTAKBANK @ ₹353.40 | Vol:0

⏰ Timeout waiting for ticks


[W 260331 22:20:26 smartWebSocketV2:319]


🎯 DAY 42: Live datafeed production!

 Attempting to resubscribe/reconnect (Attempt 1)...



• Received ticks: 4
• Processed ticks: 4
• GitHub Day42 → 42/90
✅ WebSocket opened
📡 Subscribed to: HCLTECH, KOTAKBANK, RELIANCE, HDFCBANK
📡 LIVE TICK #5: RELIANCE ₹1343.90
📡 LIVE TICK #6: HDFCBANK ₹731.55
📡 LIVE TICK #7: HCLTECH ₹1341.60
📡 LIVE TICK #8: KOTAKBANK ₹353.40
